# 04 — VASP Attribution Logic (A5)
SIH26182 — Duo A (Data & ML)

**Goal:** For any wallet, compute shortest hops to a known exchange (VASP) address.

This notebook also writes `graph_utils.py` to disk — Duo B imports `get_nearest_vasp` and the
graph object from that module for the `/score` endpoint (see B3), so keep the function signature
exactly as below.


In [2]:
import pandas as pd
import networkx as nx

edges_df = pd.read_csv("data/elliptic_txs_edgelist.csv")
G = nx.from_pandas_edgelist(
    edges_df,
    source=edges_df.columns[0],
    target=edges_df.columns[1],
    create_using=nx.DiGraph(),
)
print("Nodes:", G.number_of_nodes(), " Edges:", G.number_of_edges())


Nodes: 203769  Edges: 234355


## Known VASP placeholder addresses
The Elliptic dataset doesn't ship labeled exchange addresses, so we use clearly-labeled
placeholders for the demo — pick 5-10 real node IDs from the graph so lookups succeed.

In [3]:
import random
random.seed(42)

sample_nodes = random.sample(list(G.nodes()), 8)
known_vasps = {node: f"known_vasp_{i+1}" for i, node in enumerate(sample_nodes)}

print("known_vasps (placeholder, for demo only):")
for node_id, name in known_vasps.items():
    print(f"  {node_id} -> {name}")


known_vasps (placeholder, for demo only):
  73430399 -> known_vasp_1
  131260348 -> known_vasp_2
  231997630 -> known_vasp_3
  75462303 -> known_vasp_4
  239168246 -> known_vasp_5
  280529195 -> known_vasp_6
  234635140 -> known_vasp_7
  94466966 -> known_vasp_8


## `get_nearest_vasp` — BFS shortest path to any known VASP

In [4]:
def get_nearest_vasp(wallet_id, graph, known_vasps):
    """
    Returns (nearest_vasp_name, hop_count, confidence) for wallet_id.
    confidence: 'high' if hops<=2, 'medium' if hops<=5, else 'low'.
    If wallet_id isn't in the graph, or no path exists to any known VASP,
    returns ('unidentified', None, 'low').
    """
    if wallet_id not in graph:
        return "unidentified", None, "low"

    best_name, best_hops = None, None
    for vasp_node, vasp_name in known_vasps.items():
        if vasp_node not in graph:
            continue
        try:
            # Treat as undirected for reachability — a tx graph's directionality
            # shouldn't block a valid attribution path in either direction.
            hops = nx.shortest_path_length(graph.to_undirected(), source=wallet_id, target=vasp_node)
        except nx.NetworkXNoPath:
            continue
        if best_hops is None or hops < best_hops:
            best_hops, best_name = hops, vasp_name

    if best_name is None:
        return "unidentified", None, "low"

    if best_hops <= 2:
        confidence = "high"
    elif best_hops <= 5:
        confidence = "medium"
    else:
        confidence = "low"

    return best_name, best_hops, confidence


## Test on 3 sample wallets

In [5]:
test_wallets = random.sample(list(G.nodes()), 3)

for wallet_id in test_wallets:
    name, hops, confidence = get_nearest_vasp(wallet_id, G, known_vasps)
    print(f"Wallet {wallet_id}: nearest_vasp={name}, hops={hops}, confidence={confidence}")


Wallet 196105209: nearest_vasp=unidentified, hops=None, confidence=low
Wallet 225930984: nearest_vasp=known_vasp_2, hops=20, confidence=low
Wallet 99215359: nearest_vasp=unidentified, hops=None, confidence=low


✅ **Verify (A5):** Function runs without error and returns a hop count + VASP label (even placeholder
names) for at least one of the 3 test wallets. If most return `"unidentified"`, that's honest and expected on a
subgraph — call it out as a known limitation in the demo narrative, don't hide it.

## Export `graph_utils.py` for Duo B
This writes a standalone importable module so Duo B doesn't have to copy/paste notebook cells.

In [6]:
graph_utils_code = '''"""
graph_utils.py — shared by Duo A and Duo B.
Builds the transaction graph once and exposes get_nearest_vasp() for the /score endpoint.
"""
import pandas as pd
import networkx as nx
import random

def build_graph(edgelist_path="elliptic_txs_edgelist.csv"):
    edges_df = pd.read_csv(edgelist_path)
    graph = nx.from_pandas_edgelist(
        edges_df,
        source=edges_df.columns[0],
        target=edges_df.columns[1],
        create_using=nx.DiGraph(),
    )
    return graph


def build_known_vasps(graph, n=8, seed=42):
    random.seed(seed)
    sample_nodes = random.sample(list(graph.nodes()), n)
    return {node: f"known_vasp_{i+1}" for i, node in enumerate(sample_nodes)}


def get_nearest_vasp(wallet_id, graph, known_vasps):
    """
    Returns (nearest_vasp_name, hop_count, confidence) for wallet_id.
    confidence: \'high\' if hops<=2, \'medium\' if hops<=5, else \'low\'.
    """
    if wallet_id not in graph:
        return "unidentified", None, "low"

    best_name, best_hops = None, None
    undirected = graph.to_undirected()
    for vasp_node, vasp_name in known_vasps.items():
        if vasp_node not in graph:
            continue
        try:
            hops = nx.shortest_path_length(undirected, source=wallet_id, target=vasp_node)
        except nx.NetworkXNoPath:
            continue
        if best_hops is None or hops < best_hops:
            best_hops, best_name = hops, vasp_name

    if best_name is None:
        return "unidentified", None, "low"

    confidence = "high" if best_hops <= 2 else ("medium" if best_hops <= 5 else "low")
    return best_name, best_hops, confidence
'''

with open("graph_utils.py", "w") as f:
    f.write(graph_utils_code)

print("Wrote graph_utils.py")


Wrote graph_utils.py


---
**CHECKPOINT 2 → share with Duo B now:**
- `graph_utils.py` (contains `build_graph`, `build_known_vasps`, `get_nearest_vasp`)
- `model_v2.pkl`
- `merged_data_v2.csv` (has the graph features B3 may want for `top_features`)

This is everything Duo B needs to finish B3.
